# Apex Analytics: E-Commerce Customer Analytics & Segmentation
### Portfolio Project for Data Analyst Position

This notebook demonstrates the exploratory data analysis (EDA), customer segmentation using RFM analysis, and monthly cohort retention tracking on transactional retail data.

## 1. Setup and Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

# Design settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Clean Dataset

In [ ]:
# Load transactional data
df = pd.read_csv('data/raw_transactions.csv')
print(f"Initial shape: {df.shape}")
df.info()

In [ ]:
# Drop rows missing CustomerID
df = df.dropna(subset=['CustomerID'])
# Convert dates to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
# Filter out negative quantities (returns)
df_sales = df[df['Quantity'] > 0].copy()
df_sales['TotalSpend'] = df_sales['Quantity'] * df_sales['UnitPrice']
print(f"Cleaned sales data shape: {df_sales.shape}")
df_sales.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Plot Monthly Revenue Trends
df_sales['InvoiceMonth'] = df_sales['InvoiceDate'].dt.to_period('M')
monthly_revenue = df_sales.groupby('InvoiceMonth')['TotalSpend'].sum().reset_index()
monthly_revenue['InvoiceMonth'] = monthly_revenue['InvoiceMonth'].astype(str)

sns.lineplot(data=monthly_revenue, x='InvoiceMonth', y='TotalSpend', marker='o', color='#3b82f6', linewidth=2.5)
plt.title('Monthly E-Commerce Revenue Trends')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 Best Selling Products
top_products = df_sales.groupby('Description')['TotalSpend'].sum().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top_products, x='TotalSpend', y='Description', palette='Blues_r')
plt.title('Top 10 Selling Products by Revenue')
plt.xlabel('Total Revenue Generated ($)')
plt.ylabel('Product Description')
plt.tight_layout()
plt.show()

## 4. Customer RFM Segmentation

In [ ]:
# Recency, Frequency, Monetary calculations
reference_date = df_sales['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_sales.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalSpend': 'sum'
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSpend': 'Monetary'
}).reset_index()

rfm.head()

In [ ]:
# Divide into quintiles (scores 1-5)
rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Customer Segment categorization rule
def segment_customer(row):
    r = row['R']
    f = row['F']
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f <= 2:
        return 'Promising/New'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    else:
        return 'Lost/Hibernating'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm.head()

In [ ]:
# Visualize Customer Segments
segment_counts = rfm['Segment'].value_counts().reset_index()
sns.barplot(data=segment_counts, x='count', y='Segment', palette='viridis')
plt.title('Customer Segment Distribution')
plt.xlabel('Number of Customers')
plt.ylabel('Segment')
plt.show()

## 5. Cohort Retention Matrix

In [ ]:
# Find first purchase month
df_sales['CohortMonth'] = df_sales.groupby('CustomerID')['InvoiceMonth'].transform('min')
# Calculate CohortIndex
df_sales['CohortIndex'] = (df_sales['InvoiceMonth'].dt.year - df_sales['CohortMonth'].dt.year) * 12 + \
                           (df_sales['InvoiceMonth'].dt.month - df_sales['CohortMonth'].dt.month) + 1

# Pivot to create matrix
cohort_group = df_sales.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_matrix = cohort_group.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')

# Convert to percentage retention
cohort_sizes = cohort_matrix.iloc[:, 0]
retention_matrix = cohort_matrix.divide(cohort_sizes, axis=0) * 100
retention_matrix = retention_matrix.round(1)

# Plot Heatmap
plt.figure(figsize=(14, 8))
sns.heatmap(retention_matrix, annot=True, fmt='.1f', cmap='Blues', cbar_kws={'label': 'Retention Rate (%)'})
plt.title('Monthly Customer Retention Cohorts')
plt.xlabel('Cohort Index (Months)')
plt.ylabel('Cohort Signup Month')
plt.show()

## 6. Strategic Business Insights
1. **Champions & Loyal Customers** generate the majority of the revenue. Loyalty programs and exclusive offers should target this core group.
2. **Promising/New** customers should be nurtured with onboarding offers and active follow-up email campaigns to increase their lifetime value.
3. **At Risk** customers are previous frequent buyers that haven't shopped recently. Re-engagement campaigns with deep discounts are recommended.